# MU Escape — notatnik lokalny

Notatnik uruchamia scenariusze z pakietu 	hesis_code/ (patrz README w katalogu glownym).

**Kolejnosc:** Setup (komorki 1-2) -> scenariusze 1-5 zgodnie z opisem w komorkach.

Uruchamianie: zaznacz komorke i Shift+Enter.


---
## Setup — uruchom raz na początku sesji

### 1 — Ścieżka projektu

In [ ]:
%matplotlib inline
import os, sys

PROJECT_ROOT = os.path.abspath('..')
if not os.path.isdir(os.path.join(PROJECT_ROOT, 'thesis_code')):
    PROJECT_ROOT = os.getcwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print('Projekt:', PROJECT_ROOT)

### 2 — PyTorch i GPU

In [ ]:
import torch
from thesis_code.storage import setup_paths
from thesis_code.config import get_device

setup_paths(PROJECT_ROOT)
DEVICE = get_device()
print('PyTorch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

---
## Scenariusz 1 — CIFAR-10, quick (1 ziarno, ~1,5–2 h)

Szybki test wszystkich metod. Uruchom komórki **1.1 → 1.2 → 1.3 → 1.4**.

### 1.1 — Konfiguracja scenariusza 1

In [ ]:
from thesis_code.config import CFG, set_seed
from thesis_code.storage import ensure_dataset_dirs

CFG['dataset'] = 'CIFAR10'
QUICK = True
RESUME = True
SEEDS = [CFG['seed']] if QUICK else CFG['seeds']
set_seed(CFG['seed'])
ensure_dataset_dirs()
print('Scenariusz 1 | CIFAR10 | quick | ziarna:', SEEDS)

### 1.2 — Status checkpointów (scenariusz 1)

In [ ]:
from thesis_code.experiments import print_resume_status
print_resume_status(SEEDS)

### 1.3 — Eksperyment (scenariusz 1)

In [ ]:
import time
from thesis_code.config import CFG
from thesis_code.experiments import empty_results, merge_results, run_all_methods_for_seed

all_results_s1 = empty_results()
t0 = time.time()
for seed in SEEDS:
    print(f'\n===== Scenariusz 1 | ziarno {seed} | {CFG["dataset"]} =====')
    partial = run_all_methods_for_seed(seed, DEVICE, resume=RESUME)
    all_results_s1 = merge_results(all_results_s1, partial)
print(f'Scenariusz 1 zakończony: {(time.time()-t0)/60:.1f} min')

===== Scenariusz 1 | ziarno 42 | CIFAR10 =====
Zbiór: CIFAR10 | train: 45000 | val: 5000 | test: 10000 | ul: 2250 | klas: 10
  ↩ pominięto (już na Drive): Early Stopping seed=42
Zbiór: CIFAR10 | train: 45000 | val: 5000 | test: 10000 | ul: 2250 | klas: 10

--- Ziarno 42: SGDR ---
  SGDR epoka  10 | tr_acc=0.892 | lr=0.10000
  SGDR epoka  20 | tr_acc=0.891 | lr=0.05000
  SGDR epoka  30 | tr_acc=0.987 | lr=0.10000
  SGDR epoka  40 | tr_acc=0.884 | lr=0.08536
  SGDR epoka  50 | tr_acc=0.920 | lr=0.05000
  SGDR epoka  60 | tr_acc=0.978 | lr=0.01464
  SGDR epoka  70 | tr_acc=0.999 | lr=0.10000
  SGDR epoka  80 | tr_acc=0.888 | lr=0.09619
  SGDR → test_acc=0.8406 | sharpness=0.0631
Zbiór: CIFAR10 | train: 45000 | val: 5000 | test: 10000 | ul: 2250 | klas: 10

--- Ziarno 42: SWA ---
  SWA epoka  10 | tr_acc=0.864
  SWA epoka  20 | tr_acc=0.922
  SWA epoka  30 | tr_acc=0.964
  SWA epoka  40 | tr_acc=0.996
  SWA epoka  50 | tr_acc=0.999 [SWA]
  SWA epoka  60 | tr_acc=0.999 [SWA]
  SWA epoka  70 | tr_acc=0.959 [SWA]
  SWA epoka  80 | tr_acc=0.938 [SWA]
  Przeliczanie BatchNorm dla modelu SWA...
  SWA → test_acc=0.9414 | sharpness=0.0694
Zbiór: CIFAR10 | train: 45000 | val: 5000 | test: 10000 | ul: 2250 | klas: 10

--- Ziarno 42: Noise Injection ---
  Noise Injection: szum std=0.005
  Sharpness po szumie: 0.0329
    FT epoka   5 | tr_acc=0.917 | vl_acc=0.887
    FT epoka  10 | tr_acc=0.943 | vl_acc=0.889
    FT epoka  15 | tr_acc=0.964 | vl_acc=0.900
    FT epoka  20 | tr_acc=0.983 | vl_acc=0.909
    FT epoka  25 | tr_acc=0.992 | vl_acc=0.909
    FT epoka  30 | tr_acc=0.993 | vl_acc=0.911
  Po FT → test_acc=0.9111 | sharpness=0.1895
Zbiór: CIFAR10 | train: 45000 | val: 5000 | test: 10000 | ul: 2250 | klas: 10

--- Ziarno 42: MU Escape ---
  === MU Escape: Gradient Ascent ===
  Przed GA: vl_acc=0.7862 | sharpness=0.0421
    krok   0 | loss=0.4642 | vl_acc=0.7970
    krok   5 | loss=0.6001 | vl_acc=0.7966
    krok  10 | loss=0.3989 | vl_acc=0.7944
    krok  15 | loss=0.4753 | vl_acc=0.7940
  Po GA: sharpness=0.0418 (było 0.0421)
  === MU Escape: Fine-tuning ===
    FT epoka   5 | tr_acc=0.921 | vl_acc=0.889
    FT epoka  10 | tr_acc=0.945 | vl_acc=0.892
    FT epoka  15 | tr_acc=0.967 | vl_acc=0.901
    FT epoka  20 | tr_acc=0.982 | vl_acc=0.909
    FT epoka  25 | tr_acc=0.991 | vl_acc=0.908
    FT epoka  30 | tr_acc=0.994 | vl_acc=0.909
  Po FT → test_acc=0.9098 | sharpness=0.1962

### 1.4 — Tabela + wykresy (scenariusz 1)

In [ ]:
from IPython.display import display
from thesis_code.visualize import save_and_show_results

paths_s1 = save_and_show_results(all_results_s1, SEEDS[-1], show=True)
display(paths_s1)

---
## Scenariusz 2 — CIFAR-10, pełny protokół (3 ziarna, ~4–6 h)

Wyniki do pracy (mean ± std). Uruchom **2.1 → 2.2 → 2.3 → 2.4**.

### 2.1 — Konfiguracja scenariusza 2

In [ ]:
from thesis_code.config import CFG, set_seed
from thesis_code.storage import ensure_dataset_dirs

CFG['dataset'] = 'CIFAR10'
QUICK = False
RESUME = True
SEEDS = [CFG['seed']] if QUICK else CFG['seeds']
set_seed(CFG['seed'])
ensure_dataset_dirs()
print('Scenariusz 2 | CIFAR10 | 3 ziarna | ziarna:', SEEDS)

### 2.2 — Status checkpointów (scenariusz 2)

In [ ]:
from thesis_code.experiments import print_resume_status
print_resume_status(SEEDS)

### 2.3 — Eksperyment (scenariusz 2)

In [ ]:
import time
from thesis_code.config import CFG
from thesis_code.experiments import empty_results, merge_results, run_all_methods_for_seed

all_results_s2 = empty_results()
t0 = time.time()
for seed in SEEDS:
    print(f'\n===== Scenariusz 2 | ziarno {seed} | {CFG["dataset"]} =====')
    partial = run_all_methods_for_seed(seed, DEVICE, resume=RESUME)
    all_results_s2 = merge_results(all_results_s2, partial)
print(f'Scenariusz 2 zakończony: {(time.time()-t0)/60:.1f} min')

### 2.4 — Tabela + wykresy (scenariusz 2)

In [ ]:
from IPython.display import display
from thesis_code.visualize import save_and_show_results

paths_s2 = save_and_show_results(all_results_s2, SEEDS[-1], show=True)
display(paths_s2)

---
## Scenariusz 3 — CIFAR-100, pełny protokół (3 ziarna, ~4–6 h)

Uruchom **3.1 → 3.2 → 3.3 → 3.4**.

### 3.1 — Konfiguracja scenariusza 3

In [ ]:
from thesis_code.config import CFG, set_seed
from thesis_code.storage import ensure_dataset_dirs

CFG['dataset'] = 'CIFAR100'
QUICK = False
RESUME = True
SEEDS = [CFG['seed']] if QUICK else CFG['seeds']
set_seed(CFG['seed'])
ensure_dataset_dirs()
print('Scenariusz 3 | CIFAR100 | 3 ziarna | ziarna:', SEEDS)

### 3.2 — Status checkpointów (scenariusz 3)

In [ ]:
from thesis_code.experiments import print_resume_status
print_resume_status(SEEDS)

### 3.3 — Eksperyment (scenariusz 3)

In [ ]:
import time
from thesis_code.config import CFG
from thesis_code.experiments import empty_results, merge_results, run_all_methods_for_seed

all_results_s3 = empty_results()
t0 = time.time()
for seed in SEEDS:
    print(f'\n===== Scenariusz 3 | ziarno {seed} | {CFG["dataset"]} =====')
    partial = run_all_methods_for_seed(seed, DEVICE, resume=RESUME)
    all_results_s3 = merge_results(all_results_s3, partial)
print(f'Scenariusz 3 zakończony: {(time.time()-t0)/60:.1f} min')

### 3.4 — Tabela + wykresy (scenariusz 3)

In [ ]:
from IPython.display import display
from thesis_code.visualize import save_and_show_results

paths_s3 = save_and_show_results(all_results_s3, SEEDS[-1], show=True)
display(paths_s3)

---
## Scenariusz 4 — Diagnostyka patience (rozdz. 6)

ES z  \in \{3, 5, 7, 15\}$ — **najpierw CIFAR-10, potem CIFAR-100**.

Idź po kolei: **4.1 → 4.2 → 4.3 → 4.4 → 4.5 → 4.6** (każda komórka = Shift+Enter).

Szacowany czas: ~1,5–2,5 h (CIFAR-10) + ~2–3 h (CIFAR-100) na RTX 3050.

### 4.1 — CIFAR-10: konfiguracja

In [ ]:
from thesis_code.config import CFG, set_seed
from thesis_code.storage import ensure_dataset_dirs

CFG['dataset'] = 'CIFAR10'
PATIENCE_SEED = 42
set_seed(PATIENCE_SEED)
ensure_dataset_dirs()
print('Scenariusz 4.1 | CIFAR-10 | seed:', PATIENCE_SEED)

### 4.2 — CIFAR-10: eksperyment patience (~1,5–2,5 h)

In [ ]:
import time
from thesis_code.experiments import run_patience_diagnostic

t0 = time.time()
df_patience_c10 = run_patience_diagnostic(
    DEVICE, num_classes=10,
    seed=PATIENCE_SEED,
    verbose=True,
    show=True,
)
print(f'CIFAR-10 zakończono w {(time.time()-t0)/60:.1f} min')
display(df_patience_c10)

### 4.3 — CIFAR-10: podgląd wyników

In [ ]:
from IPython.display import Image, display
import pandas as pd
import os

base_c10 = os.path.join(PROJECT_ROOT, 'results', 'CIFAR10')
print('CSV:', os.path.join(base_c10, 'diagnostyka_patience.csv'))
display(pd.read_csv(os.path.join(base_c10, 'diagnostyka_patience.csv')))
display(Image(filename=os.path.join(base_c10, 'diagnostyka_sharpness.png'), width=900))

### 4.4 — CIFAR-100: konfiguracja

In [ ]:
from thesis_code.config import CFG, set_seed
from thesis_code.storage import ensure_dataset_dirs

CFG['dataset'] = 'CIFAR100'
set_seed(PATIENCE_SEED)
ensure_dataset_dirs()
print('Scenariusz 4.4 | CIFAR-100 | seed:', PATIENCE_SEED)

### 4.5 — CIFAR-100: eksperyment patience (~2–3 h)

In [ ]:
import time
from thesis_code.experiments import run_patience_diagnostic

t0 = time.time()
df_patience_c100 = run_patience_diagnostic(
    DEVICE, num_classes=100,
    seed=PATIENCE_SEED,
    verbose=True,
    show=True,
)
print(f'CIFAR-100 zakończono w {(time.time()-t0)/60:.1f} min')
display(df_patience_c100)

### 4.6 — CIFAR-100: podgląd wyników

In [ ]:
from IPython.display import Image, display
import pandas as pd
import os

base_c100 = os.path.join(PROJECT_ROOT, 'results', 'CIFAR100')
print('CSV:', os.path.join(base_c100, 'diagnostyka_patience.csv'))
display(pd.read_csv(os.path.join(base_c100, 'diagnostyka_patience.csv')))
display(Image(filename=os.path.join(base_c100, 'diagnostyka_sharpness.png'), width=900))

---
## Scenariusz 5 — Przeliczenie wykresów (białe tło, panele osobno)

**Bez treningu** — wczytuje checkpointy i generuje:
- porownanie_wykres.png (3 panele razem)
- porownanie_val_acc.png, porownanie_train_loss.png, porownanie_sharpness.png (osobno)
- `wykres_finalny.png`, `tabela_finalna.csv`, `trajektorie_seed777.json`

Uruchom **5.1** dla CIFAR-10, potem **5.2** dla CIFAR-100 (~5–10 min łącznie).

### 5.1 — Przelicz wykresy CIFAR-10

In [ ]:
from thesis_code.config import CFG
from thesis_code.visualize import regenerate_plots_from_checkpoints

CFG['dataset'] = 'CIFAR10'
paths_c10 = regenerate_plots_from_checkpoints([42, 123, 777], show=True)
display(paths_c10)

### 5.2 — Przelicz wykresy CIFAR-100

In [ ]:
CFG['dataset'] = 'CIFAR100'
paths_c100 = regenerate_plots_from_checkpoints([42, 123, 777], show=True)
display(paths_c100)

---
## Na koniec — podgląd wszystkich wyników z dysku

In [ ]:
import pandas as pd
from IPython.display import Image, display
import os

for ds in ['CIFAR10', 'CIFAR100']:
    base = os.path.join(PROJECT_ROOT, 'results', ds)
    print('\n' + '='*60, ds, '='*60)
    if os.path.isfile(os.path.join(base, 'tabela_finalna.csv')):
        display(pd.read_csv(os.path.join(base, 'tabela_finalna.csv')))
    for png in [
        'wykres_finalny.png',
        'porownanie_wykres.png',
        'porownanie_val_acc.png',
        'porownanie_train_loss.png',
        'porownanie_sharpness.png',
        'diagnostyka_sharpness.png',
    ]:
        p = os.path.join(base, png)
        if os.path.isfile(p):
            print(png)
            display(Image(filename=p, width=700))
        else:
            print('(brak)', png)
